### Contract Knowledge Agent

Creates an Agent Bricks Knowledge Assistant over the vendor supply contract PDFs stored
in the Unity Catalog volume. Answers questions about payment terms, fixed price schedules,
volume discount tiers, SLA requirements, penalty clauses, and dispute resolution procedures.

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
ENDPOINT_NAME = dbutils.widgets.get("CONTRACT_KNOWLEDGE_ENDPOINT_NAME")

In [ ]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

VOLUME_PATH = f"/Volumes/{CATALOG}/procurement/contracts"
AGENT_NAME = f"{CATALOG}-contract-knowledge"
API_BASE = "/api/2.0/knowledge-assistants"

KA_BODY = {
    "name": AGENT_NAME,
    "description": (
        "Answers questions about Caspers Kitchens vendor supply contracts including "
        "payment terms, contracted unit prices, volume discount schedules, delivery SLA "
        "commitments, late payment penalty rates, price stability clauses, and dispute "
        "resolution procedures. Covers 10 supplier contracts across all procurement categories."
    ),
    "endpoint_name": ENDPOINT_NAME,
    "knowledge_sources": [
        {
            "files_source": {
                "name": "contract_pdfs",
                "type": "files",
                "files": {"path": VOLUME_PATH},
                "description": (
                    "Vendor supply agreement PDFs for Caspers Kitchens. "
                    "Each contract covers: parties, contract term (effective and expiration dates), "
                    "payment terms (Net 30 / Net 45 / Net 21 / 2/10 Net 30), "
                    "Schedule A price schedule with fixed unit prices per item, "
                    "volume discount tiers (where applicable), delivery SLA commitments "
                    "and per-day penalty rates for late delivery, "
                    "late payment penalty rates (typically 1.5%/month), "
                    "price stability clauses prohibiting unilateral price increases, "
                    "and dispute resolution procedures."
                ),
            },
        },
    ],
    "instructions": (
        "You are a contract analyst for Caspers Kitchens. "
        "Always cite the contract number (e.g. CK-VFP-2023-001) and supplier name when answering. "
        "Be precise about contract terms — exact prices, percentages, day counts, and thresholds. "
        "When a question involves whether an invoice is compliant with a contract, "
        "state the contract term explicitly and flag any deviation as a potential breach. "
        "Quote directly from the contract text where relevant."
    ),
}

def get_tile_by_name(name):
    resp = w.api_client.do("GET", "/api/2.0/tiles")
    for tile in resp.get("tiles", []):
        if tile.get("name") == name:
            return tile
    return None

def find_existing_id():
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type IN ('knowledge_assistants', 'endpoints')
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            info = json.loads(row.resource_data)
            if info.get("endpoint_name") == ENDPOINT_NAME:
                return info.get("tile_id") or info.get("agent_id")
    except Exception:
        pass
    return None

def try_get(agent_ref):
    try:
        return w.api_client.do("GET", f"{API_BASE}/{agent_ref}")
    except Exception:
        return None

def try_delete(agent_ref):
    try:
        w.api_client.do("DELETE", f"{API_BASE}/{agent_ref}")
        print(f"  Deleted stale Knowledge Assistant: {agent_ref}")
    except Exception as e:
        print(f"  DELETE failed (ignoring): {e}")

def try_put(agent_ref):
    try:
        w.api_client.do("PUT", f"{API_BASE}/{agent_ref}", body=KA_BODY)
        return True
    except Exception as e:
        print(f"  PUT failed ({type(e).__name__}: {e}) — will delete and recreate.")
        try_delete(agent_ref)
        return False

existing_id = find_existing_id()
agent_id = None
needs_polling = True

# Try updating via uc_state id
if existing_id:
    info = try_get(existing_id)
    if info:
        print(f"Knowledge Assistant exists ({existing_id}), updating...")
        if try_put(existing_id):
            agent_id = existing_id
            needs_polling = False
            print(f"\u2705 Updated Knowledge Assistant: {agent_id}")

# Try updating via name lookup (if uc_state update failed or had no id)
if not agent_id:
    tile = get_tile_by_name(AGENT_NAME)
    if tile:
        tile_id = tile["tile_id"]
        print(f"\u267b\ufe0f Found Knowledge Assistant by name: {tile_id}")
        if try_put(tile_id):
            agent_id = tile_id
            needs_polling = False
            print(f"\u2705 Updated Knowledge Assistant: {agent_id}")

# Create fresh if update/delete cleared the way
if not agent_id:
    try:
        w.api_client.do("POST", API_BASE, body=KA_BODY)
        print(f"\u2705 Created Knowledge Assistant")
    except Exception as e:
        err = str(e)
        if "already exists" in err.lower() or "ALREADY_EXISTS" in err:
            needs_polling = False
            print(f"\u267b\ufe0f Agent {AGENT_NAME} already exists. Proceeding.")
        else:
            raise

# Resolve tile_id
if not agent_id:
    tile = get_tile_by_name(AGENT_NAME)
    if tile:
        agent_id = tile["tile_id"]
        print(f"Resolved tile_id: {agent_id}")
    else:
        agent_id = AGENT_NAME
        needs_polling = False
        print(f"\u26a0\ufe0f Could not resolve tile_id, using name as fallback: {agent_id}")

print(f"   Agent ID (tile_id): {agent_id}")
print(f"   Endpoint: {ENDPOINT_NAME}")

In [ ]:
import time

if needs_polling and agent_id:
    MAX_WAIT = 300
    POLL_INTERVAL = 30
    elapsed = 0
    poll_ref = agent_id if agent_id != AGENT_NAME else AGENT_NAME
    print(f"Checking if Knowledge Assistant is ready (ref={poll_ref}, max {MAX_WAIT}s)...")

    while elapsed < MAX_WAIT:
        try:
            ka_status = w.api_client.do("GET", f"{API_BASE}/{poll_ref}")
            ep_status = ka_status["knowledge_assistant"]["status"].get("endpoint_status", "")
            print(f"  endpoint_status: {ep_status}")
            if str(ep_status).upper() in ("ONLINE", "ACTIVE", "READY"):
                print(f"\u2705 Knowledge Assistant {AGENT_NAME} is READY")
                break
        except Exception as e:
            print(f"  GET status check failed: {type(e).__name__}: {e}")

        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
    else:
        print(f"\u23f3 Endpoint may still be provisioning — proceeding.")
else:
    print(f"\u2705 Knowledge Assistant already running — skipping polling.")

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

add(CATALOG, "knowledge_assistants", {"endpoint_name": ENDPOINT_NAME, "tile_id": agent_id, "name": AGENT_NAME})
print("\u2705 Contract Knowledge Agent stage complete")